In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import json
import joblib
import shutil
import sys

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
 
if str(SRC_PATH) not in sys.path:
     sys.path.insert(0, str(SRC_PATH))
 
print("Projet :", PROJECT_ROOT)
print("Src    :", SRC_PATH)

In [ ]:
data = df_preprocessed(as_frame=True)
df = data.frame
 
X = df.drop(columns=["evo_conso_scaled"])
y = df["evo_conso_scaled"]
 
X_train, X_test, y_train, y_test = train_test_split(
     X,
     y,
     test_size=0.2,
     random_state=42,
 )
 
df.head()

In [ ]:
def train_model(X_train, y_train):
    """Entraîne trois modèles de régression et retourne le meilleur
    basé sur la MAE en cross-validation."""
    model_v1 = {
        "RandomForest": RandomForestRegressor(
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        "LinearRegression": LinearRegression(),
        "GradientBoosting": GradientBoostingRegressor(random_state=42),
    }

    best_model = None
    best_score = -np.inf

    for _, model in model_v1.items():
        scores = cross_val_score(
            model, X_train, y_train, cv=5, scoring="neg_mean_absolute_error"
        )
        mean_score = scores.mean()
        if mean_score > best_score:
            best_score = mean_score
            best_model = model

    # Entraîne le meilleur modèle sur l'ensemble des données d'entraînement
    best_model.fit(X_train, y_train)
    print(f"Meilleur modèle sélectionné : {type(best_model).__name__}")
    return best_model

In [ ]:
def evaluate(model, X_test, y_test):
     predictions = model.predict(X_test)
     return {
         "mae": float(mean_absolute_error(y_test, predictions)),
         "rmse": float(np.sqrt(mean_squared_error(y_test, predictions))),
         "r2": float(r2_score(y_test, predictions)),
     }
 
metrics_v1 = evaluate(model_v1, X_test, y_test)
metrics_v1

In [ ]:
models_dir = PROJECT_ROOT / "artifacts" / "models"
metrics_dir = PROJECT_ROOT / "artifacts" / "metrics"
models_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)
 
joblib.dump(model_v1, models_dir / "model_v1.joblib")
 
with open(metrics_dir / "metrics_v1.json", "w", encoding="utf-8") as file:
     json.dump(metrics_v1, file, indent=2)
 
print("Modele V1 sauvegarde")

In [ ]:
def train_model(X_train, y_train):
    """Entraîne trois modèles de régression et retourne le meilleur
    basé sur la MAE en cross-validation."""
    model_v2 = {
        "RandomForest": RandomForestRegressor(
            n_estimators=100, random_state=42, n_jobs=-1
        ),
        "LinearRegression": LinearRegression(),
        "GradientBoosting": GradientBoostingRegressor(random_state=42),
    }

    best_model = None
    best_score = -np.inf

    for _, model in model_v2.items():
        scores = cross_val_score(
            model, X_train, y_train, cv=5, scoring="neg_mean_absolute_error"
        )
        mean_score = scores.mean()
        if mean_score > best_score:
            best_score = mean_score
            best_model = model

    # Entraîne le meilleur modèle sur l'ensemble des données d'entraînement
    best_model.fit(X_train, y_train)
    print(f"Meilleur modèle sélectionné : {type(best_model).__name__}")
    return best_model


In [ ]:
joblib.dump(model_v2, models_dir / "model_v2.joblib")
 
with open(metrics_dir / "metrics_v2.json", "w", encoding="utf-8") as file:
     json.dump(metrics_v2, file, indent=2)
 
print("Modele V2 sauvegarde")

In [ ]:
# Comparer les deux modèles et choisir le meilleur
print("\n" + "="*50)
print("      COMPARAISON DES MODÈLES V1 ET V2")
print("="*50)

print("\nModèle V1:")
print(f"  MAE  : {metrics_v1['mae']:.4f}")
print(f"  RMSE : {metrics_v1['rmse']:.4f}")
print(f"  R²   : {metrics_v1['r2']:.4f}")

print("\nModèle V2:")
print(f"  MAE  : {metrics_v2['mae']:.4f}")
print(f"  RMSE : {metrics_v2['rmse']:.4f}")
print(f"  R²   : {metrics_v2['r2']:.4f}")

# Choisir le meilleur modèle basé sur la MAE (plus basse = meilleur)
best_model_name = "v1" if metrics_v1["mae"] < metrics_v2["mae"] else "v2"
best_metrics = metrics_v1 if best_model_name == "v1" else metrics_v2
best_model_final = model_v1 if best_model_name == "v1" else model_v2

print("\n" + "="*50)
print(f"✓ Meilleur modèle: {best_model_name.upper()}")
print(f"  MAE  : {best_metrics['mae']:.4f}")
print(f"  RMSE : {best_metrics['rmse']:.4f}")
print(f"  R²   : {best_metrics['r2']:.4f}")
print("="*50)

In [ ]:
shutil.copy(
     models_dir / "best_model_name.joblib",
     models_dir / "model_latest.joblib",
 )
 
with open(metrics_dir / "best_metrics.json", "r", encoding="utf-8") as src:
     latest_metrics = json.load(src)
 
with open(metrics_dir / "metrics_latest.json", "w", encoding="utf-8") as dst:
     json.dump(latest_metrics, dst, indent=2)
 
print("Modele actif : model_latest.joblib")


In [ ]:
list((PROJECT_ROOT / "artifacts" / "models").iterdir()), list((PROJECT_ROOT / "artifacts" / "metrics").iterdir())